In [1]:
from torch_geometric.datasets import Planetoid, WebKB, WikipediaNetwork
from torch_geometric import transforms as T
from torch_geometric.utils import (
    coalesce,
    to_dense_adj,dense_to_sparse,to_torch_csr_tensor,to_edge_index,add_self_loops
)
import numpy as np
import torch
from HLCL.utils import split_edges
def get_split_given(data, run, device):
    split = {}
    split["train"] = torch.where(data.train_mask.t()[run])[0].to(device)
    split["valid"] = torch.where(data.val_mask.t()[run])[0].to(device)
    split["test"] = torch.where(data.test_mask.t()[run])[0].to(device)
    return split
file_loc = './data/'
dataset_name = "chameleon"
device = torch.device("cuda:{}".format(0))
# dataset = Planetoid(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
# dataset = WebKB(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
dataset = WikipediaNetwork(root=file_loc+dataset_name, name=dataset_name, transform=T.NormalizeFeatures())
data = dataset[0]
data.edge_index = add_self_loops(data.edge_index)[0]
split = get_split_given(data, 0, device)
unknown_edges, known_edges = split_edges(data.edge_index, split)

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
data.edge_index.shape[0]

2

In [7]:
graph = data.x @ data.x.t()
adj_idx = to_dense_adj(unknown_edges.t())[0]
graph = graph*adj_idx
edge_idx, edge_weight = dense_to_sparse(graph)

In [8]:
a = to_dense_adj(edge_idx)


In [11]:
a

tensor([[[0.3000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.3000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.3000,  ..., 0.0000, 0.3000, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.3000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.3000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.3000]]])

In [12]:
a[a==0] = -1
a[a>0] = 0

a = a * -1

In [14]:
a

tensor([[[-0., 1., 1.,  ..., 1., 1., 1.],
         [1., -0., 1.,  ..., 1., 1., 1.],
         [1., 1., -0.,  ..., 1., -0., 1.],
         ...,
         [1., 1., 1.,  ..., -0., 1., 1.],
         [1., 1., 1.,  ..., 1., -0., 1.],
         [1., 1., 1.,  ..., 1., 1., -0.]]])

In [22]:
edge_idx

tensor([[   0,    0,    0,  ..., 2276, 2276, 2276],
        [   0, 1667, 2156,  ..., 1860, 2212, 2276]])

In [20]:
to_dense_adj(unknown_edges.t())[0]

torch.Size([2277, 2277])

In [3]:
def th_delete(tensor, indices):
    mask = torch.ones(tensor.numel(), dtype=torch.bool)
    mask[indices] = False
    return tensor[mask], mask.nonzero().t()[0]

In [4]:
k = len(edge_idx.t())

In [5]:
node_lst = {}
weight_lst = {}
for i in range(k):
    if i > 0 and edge_idx[0][i] == edge_idx[0][i-1]:
        node_lst[edge_idx[0][i].item()].append(edge_idx[1][i])
        weight_lst[edge_idx[0][i].item()].append(edge_weight[i])
    else:
        node_lst[edge_idx[0][i].item()] = []
        node_lst[edge_idx[0][i].item()].append(edge_idx[1][i])
        weight_lst[edge_idx[0][i].item()] = []
        weight_lst[edge_idx[0][i].item()].append(edge_weight[i])

In [13]:
high_k = 0.2
low_k = 0.3
low_graph = []
low_weight = []
high_graph = []
high_weight = []
for node, wgt in weight_lst.items():
    t_len = len(wgt)
    hk = np.ceil(t_len * high_k).astype(int)
    lk = np.ceil(t_len * low_k).astype(int)
    wgt = torch.tensor(wgt)
    low_edge_wgt, low_edge_idx = torch.topk(wgt, lk)
    remained_wgts, remained_idx = th_delete(wgt, low_edge_idx)
    if len(remained_wgts) > hk:
        high_edge_wgt, high_edge_idx = torch.topk(remained_wgts, hk,largest=False)
        high_edge_idx = remained_idx[high_edge_idx]
    else:
        high_edge_wgt, high_edge_idx = remained_wgts, remained_idx
    for idx, neigh_node in enumerate(high_edge_idx):
        edge_pair = torch.tensor([node, node_lst[node][neigh_node]])
        high_graph.append(edge_pair)
        high_weight.append(high_edge_wgt[idx])
    
    for idx, neigh_node in enumerate(low_edge_idx):
        edge_pair = torch.tensor([node, node_lst[node][neigh_node]])
        low_graph.append(edge_pair)
        low_weight.append(low_edge_wgt[idx])
high_graph = torch.stack(high_graph).t()
low_graph = torch.stack(low_graph).t()
    

In [14]:
high_graph

tensor([[   0,    1,    2,  ..., 2702, 2706, 2707],
        [2582,    2,    1,  ..., 1536, 1473,  598]])

In [17]:
a = to_dense_adj(low_graph)

In [19]:
to_dense_adj(low_graph)

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 1., 0.]]])

In [18]:
a[a==0] = -1
a[a==1] = 0
a = a * -1
a

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., -0., 1.]]])

In [15]:
to_dense_adj(high_graph)

tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 1.,  ..., 0., 0., 0.],
         [0., 1., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])

In [ ]:
torch.stack(high_graph)

TypeError: expected Tensor as element 0 in argument 0, but got list

In [ ]:
torch.cat(high_graph)

TypeError: expected Tensor as element 0 in argument 0, but got list

In [ ]:
torch.stack([torch.FloatTensor([1]).view(1, -1), torch.FloatTensor([2]).view(1, -1)]

[tensor([[1.]]), tensor([[2.]])]

In [ ]:
len(high_weight)

2597

In [ ]:
l1 = ["eat", "sleep", "repeat"]

In [ ]:
for count, ele in enumerate(l1):
    print(count)
    print(ele)

0
eat
1
sleep
2
repeat
